# Compare GA and ClustalW

In [ ]:
# Import our algorithms
%run ga.ipynb

# Import Biopython for ClustalW
from Bio import SeqIO, AlignIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import subprocess
import time
import os

## Utils

In [ ]:
def run_clustalw(sequences, output_prefix="temp"):
    clustalw_path = r"C:\Program Files (x86)\ClustalW2\clustalw2.exe"
    
    fasta_file = f"{output_prefix}.fasta"
    records = [SeqRecord(Seq(seq), id=f"seq{i}", description="") 
               for i, seq in enumerate(sequences)]
    SeqIO.write(records, fasta_file, "fasta")
    
    # Run ClustalW using subprocess
    start = time.time()
    try:
        cmd = [
            clustalw_path,
            f"-INFILE={fasta_file}",
            "-TYPE=PROTEIN",
            "-MATRIX=BLOSUM",      # Uses BLOSUM62
            "-GAPOPEN=10",         # Gap open penalty 
            "-GAPEXT=1"            # Gap extend penalty 
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        elapsed = time.time() - start
        
        if result.returncode != 0:
            print(f"ClustalW error: {result.stderr}")
            return None, elapsed
    except FileNotFoundError:
        print(f"Error: clustalw2 not found at {clustalw_path}")
        print("Update the clustalw_path variable if installed elsewhere")
        return None, 0
    except Exception as e:
        print(f"Error running ClustalW: {e}")
        return None, 0
    
    # Read alignment
    aln_file = f"{output_prefix}.aln"
    try:
        alignment = AlignIO.read(aln_file, "clustal")
        aligned_seqs = [str(record.seq) for record in alignment]
    except Exception as e:
        print(f"Error reading alignment: {e}")
        return None, elapsed
    
    for ext in [".fasta", ".aln", ".dnd"]:
        try:
            os.remove(f"{output_prefix}{ext}")
        except:
            pass
    
    return aligned_seqs, elapsed

In [ ]:
def compare_algorithms(sequences, max_gaps=3, ga_configs=None):
    assert(ga_configs != None)

    print(f"Sequences: {[s[:20] + '...' if len(s) > 20 else s for s in sequences]}")
    print(f"Max gaps: {max_gaps}")
    print()

    result = {
        'sequences': sequences,
        'max_gaps': max_gaps
    }

    # Run each GA configuration
    for i, config in enumerate(ga_configs, 1):
        name = config['name']
        params = config['params']

        print(f"=== {name} ===")
        print(f"Params: pop={params['population_size']}, gen={params['num_generations']}, "
              f"tour={params['tournament_size']}, mut={params['mutation_prob']}, elite={params['elitism_size']}")

        start = time.time()
        ga_individual, _ = ga_msa(sequences, max_gaps=max_gaps, **params)
        ga_time = time.time() - start
        ga_score = ga_individual.fitness

        print(f"Score: {ga_score:.2f}, Time: {ga_time:.3f}s")

        result[f'{name}_score'] = ga_score
        result[f'{name}_time'] = ga_time
        result[f'{name}_alignment'] = ga_individual.code
        print()

    # Run ClustalW
    print("=== CLUSTALW ===")
    clustalw_alignment, clustalw_time = run_clustalw(sequences)
    if clustalw_alignment:
        clustalw_score = calculate_sp_score(clustalw_alignment)
        print(f"Score: {clustalw_score:.2f}, Time: {clustalw_time:.3f}s")
    else:
        clustalw_score = None
        clustalw_time = None
        print("ClustalW failed to run")

    result['ClustalW_score'] = clustalw_score
    result['ClustalW_time'] = clustalw_time
    result['ClustalW_alignment'] = clustalw_alignment
    print()

    # Summary
    print("=== SUMMARY ===")
    for config in ga_configs:
        name = config['name']
        print(f"{name:12} Score={result[f'{name}_score']:>10.2f}, Time={result[f'{name}_time']:>8.3f}s")

    if clustalw_score is not None:
        print(f"{'ClustalW':12} Score={clustalw_score:>10.2f}, Time={clustalw_time:>8.3f}s")
    print("="*60)

    return result

## Testing

In [ ]:
test_cases = [
    # Original test cases
    (['ACT', 'ACT', 'ACT'], 3),
    (['ACT', 'AT', 'ACT'], 3),
    (['AG', 'A', 'AG'], 3),
    (['ACGT', 'AGT', 'AT'], 3),
    (['GAT', 'AT', 'T'], 3),
    (['GATTACA', 'GTACA', 'GACA'], 3),
    (['ATCG', 'ACG', 'ATG'], 3),
    (['CGTA', 'CTA', 'CGA'], 3),
    (['ACE', 'AE', 'CE'], 3),
    (['PAWHE', 'HEAWY', 'AWHE'], 3),
    (['ACDEG', 'ACEG', 'ADEG'], 3),
    (['PAWH', 'AWH', 'PWH', 'AH'], 3),
    (['ACGT', 'AGT'], 3),
    (['HELIX', 'HEAX'], 3),
    (['A', 'C', 'T'], 3),
    (['W', 'H', 'E'], 3),
    (['ATGATG', 'ATGTG', 'ATG'], 3),
    (['ABCD', 'BCD', 'CD', 'D'], 3),
    (['GATTACAGAT', 'GATTGAT', 'GACAGAT', 'GATTAGAT'], 3),
    (['HELICASE', 'HELCASE', 'HECASE', 'HEASE'], 3),
]

In [ ]:
%run balibase_rv11_subset.py

In [ ]:
# Define your 3 GA configurations here
ga_configs = [
    {
        'name': 'GA_Small',
        'params': {
            'population_size': 50,
            'num_generations': 30,
            'tournament_size': 5,
            'mutation_prob': 0.1,
            'elitism_size': 5
        }
    },
    {
        'name': 'GA_Medium',
        'params': {
            'population_size': 100,
            'num_generations': 50,
            'tournament_size': 10,
            'mutation_prob': 0.1,
            'elitism_size': 10
        }
    },
    {
        'name': 'GA_Large',
        'params': {
            'population_size': 150,
            'num_generations': 100,
            'tournament_size': 20,
            'mutation_prob': 0.05,
            'elitism_size': 20
        }
    }
]

# Run tests
results = []
for i, (seqs, max_gaps) in enumerate(test_cases_rv11, 1):
    print(f"\n{'='*60}")
    print(f"TEST {i}/{len(test_cases_rv11)}")
    print(f"{'='*60}\n")
    result = compare_algorithms(seqs, max_gaps=max_gaps, ga_configs=ga_configs)
    results.append(result)
    print("\n")

## Results Summary

In [16]:
import pandas as pd

if results:
    ga_names = [key.replace('_score', '') for key in results[0].keys() if key.endswith('_score') and key != 'ClustalW_score']
else:
    ga_names = ['GA_Small', 'GA_Medium', 'GA_Large']

df = pd.DataFrame(results)
df['test_num'] = range(1, len(df) + 1)
df['seq_str'] = df['sequences'].apply(lambda x: ', '.join([s[:10] + '...' if len(s) > 10 else s for s in x[:2]]) + ('...' if len(x) > 2 else ''))

print("\n" + "="*150)
print("OVERALL SUMMARY - MULTIPLE GA CONFIGURATIONS")
print("="*150)

header = f"{'Test':<5} {'Sequences':<30}"
for ga_name in ga_names:
    header += f" {ga_name:>12} {'Time':>10}"
header += f" {'ClustalW':>12} {'Time':>10}"
print(header)

subheader = f"{'':5} {'':30}"
for ga_name in ga_names:
    subheader += f" {'Score':>12} {'':10}"
subheader += f" {'Score':>12} {'':10}"
print(subheader)
print("-"*150)

# Print each test result
for _, r in df.iterrows():
    seqs_str = r['seq_str'][:30]
    row = f"{int(r['test_num']):<5} {seqs_str:<30}"
    
    # Add each GA's results
    for ga_name in ga_names:
        score = r[f'{ga_name}_score']
        time_val = r[f'{ga_name}_time']
        row += f" {score:>12.2f} {time_val:>9.3f}s"
    
    # Add ClustalW results
    cw_score = r['ClustalW_score'] if pd.notna(r['ClustalW_score']) else float('nan')
    cw_time = r['ClustalW_time'] if pd.notna(r['ClustalW_time']) else float('nan')
    row += f" {cw_score:>12.2f} {cw_time:>9.3f}s"
    
    print(row)

print("-"*150)

# Calculate statistics for each GA
valid_results = df[df['ClustalW_score'].notna()]
total_tests = len(valid_results)

print(f"\n{'Algorithm':<12} {'Avg Score':>12} {'Avg Time':>10} {'vs CW Better':>15} {'vs CW Worse':>15} {'vs CW Tied':>15}")
print("-"*150)

for ga_name in ga_names:
    avg_score = df[f'{ga_name}_score'].mean()
    avg_time = df[f'{ga_name}_time'].mean()
    
    if total_tests > 0:
        ga_better = len(valid_results[valid_results[f'{ga_name}_score'] > valid_results['ClustalW_score']])
        ga_worse = len(valid_results[valid_results[f'{ga_name}_score'] < valid_results['ClustalW_score']])
        tied = len(valid_results[abs(valid_results[f'{ga_name}_score'] - valid_results['ClustalW_score']) < 0.01])
        
        print(f"{ga_name:<12} {avg_score:>12.2f} {avg_time:>9.3f}s "
              f"{ga_better:>4}/{total_tests} ({ga_better/total_tests*100:>4.1f}%) "
              f"{ga_worse:>4}/{total_tests} ({ga_worse/total_tests*100:>4.1f}%) "
              f"{tied:>4}/{total_tests} ({tied/total_tests*100:>4.1f}%)")

if total_tests > 0:
    avg_cw_score = valid_results['ClustalW_score'].mean()
    avg_cw_time = valid_results['ClustalW_time'].mean()
    print(f"{'ClustalW':<12} {avg_cw_score:>12.2f} {avg_cw_time:>9.3f}s {'—':>20} {'—':>20} {'—':>20}")

print("\n" + "="*150)

# Best GA per test
print("\nBEST GA PER TEST:")
print("-"*80)
for _, r in df.iterrows():
    scores = {ga_name: r[f'{ga_name}_score'] for ga_name in ga_names}
    best_ga = max(scores, key=scores.get)
    best_score = scores[best_ga]
    cw_score = r['ClustalW_score'] if pd.notna(r['ClustalW_score']) else float('-inf')
    
    winner = best_ga if best_score > cw_score else 'ClustalW'
    winner_score = best_score if best_score > cw_score else cw_score
    
    print(f"Test {int(r['test_num']):2d}: Best={winner:<12} (Score: {winner_score:>10.2f})")

print("="*150)


OVERALL SUMMARY - MULTIPLE GA CONFIGURATIONS
Test  Sequences                          GA_Small       Time    GA_Medium       Time     GA_Large       Time     ClustalW       Time
                                            Score                   Score                   Score                   Score           
------------------------------------------------------------------------------------------------------------------------------------------------------
1     GKGDPKKPRG..., MQDRVKRPMN.....      -576.00     0.491s      -397.00     1.486s      -249.00     4.068s       172.00     0.025s
2     NLFVALYDFV..., PLALLLDSSL.....     -4985.00     2.187s     -5392.00     6.780s     -4426.00    18.706s     -2730.00     0.035s
3     SISDTVKRAR..., MTVEPFRNEP.....     -2840.00     1.981s     -2746.00     6.344s     -2704.00    17.256s      -787.00     0.115s
4     MIKIPRGTQD..., RDHRKIGKQL.....     -3006.00     1.947s     -2227.00     6.616s     -2709.00    18.866s     -1586.00     0.102s
5    